In [1]:
import math

def nCr(n, r):
    """ Standard combination function (n choose r) """
    if r < 0 or r > n:
        return 0
    return math.comb(n, r)

def get_Eb_Ub(q, m, n, K, r, b):
    """
    Computes the number of equations (Eb) and unknowns (Ub) 
    for the Support Minors (SM) attack.
    """
    if q > 2:
        # Standard Support Minors (Bardet et al. 2020)
        Eb = sum((-1)**(i+1) * nCr(n, r+i) * nCr(m+i-1, i) * nCr(K+b-i-1, b-i) 
                 for i in range(1, b + 1))
        Ub = nCr(K+b-1, b) * nCr(n, r)
    else:
        # Improved Support Minors for q=2 (Bros et al. 2022)
        Eb = sum(sum((-1)**(i+1) * nCr(n, r+i) * nCr(m+i-1, i) * nCr(K, j-i) 
                     for i in range(1, j + 1)) for j in range(1, b + 1))
        Ub = sum(nCr(n, r) * nCr(K, j) for j in range(1, b + 1))
    return Eb, Ub

def compute_row_complexity(q, m, n, k, k_prime, delta, omega=2.38):
    """
    Evaluates the security level by comparing Decoding attacks 
    (Kernel, Support Minors) and the Structural Distinguisher (Overbeck-like).
    """
    K = k_prime + 1  
    t_pub = math.floor((n - k) / (2 * delta))
    log2_q = math.log2(q)
    
    # Storage for the lowest attack cost (best strategy for the attacker)
    min_log_work = float('inf')
    
    # --- 1. Structural Distinguisher Complexity ---
    dist_work = (k * m - k_prime) * m * log2_q
    if dist_work < min_log_work:
        min_log_work = dist_work

    # --- 2. Hybrid-Kernel Attack ---
    for a in range(math.ceil(K/m)):
        K_red = K - a*m
        n_red = n - a
        r_red = t_pub
        if K_red <= 0 or n_red <= 0 or r_red <= 0: continue
        
        exp = a*r_red + r_red*math.ceil(K_red/m)
        work_kernel = exp * log2_q + omega * math.log2(K_red)
        if work_kernel < min_log_work:
            min_log_work = work_kernel

    # --- 3. Hybrid-SM (Support Minors) Attack ---
    for a in range(math.ceil(K/m)):
        for b in range(1, t_pub + 2):
            K_red = K - a*m
            n_red = n - a
            r_red = t_pub
            if K_red <= 0 or n_red <= 0 or r_red < 0: continue
            
            Eb, Ub = get_Eb_Ub(q, m, n_red, K_red, r_red, b)
            
            if Eb >= Ub and Ub > 0:
                prefix = (a * r_red) * log2_q
                log2_Eb = math.log2(Eb)
                log2_Ub = math.log2(Ub)
                
                # We now use min(sm1, sm2) for all q >= 2
                # sm1: algebraic resolution cost
                sm1 = prefix + log2_Eb + (omega - 1) * log2_Ub
                # sm2: system construction and linear algebra cost
                sm2 = prefix + math.log2(K_red * (r_red + 1)) + log2_Eb + log2_Ub
                
                work_sm = min(sm1, sm2)
                
                if work_sm < min_log_work:
                    min_log_work = work_sm

    # --- Size Calculations ---
    pk_kb = (k_prime * (m*n - k_prime) * log2_q) / (8 * 1000)
    ct_b = ((m*n - k_prime) * log2_q) / 8
    
    return t_pub, min_log_work, pk_kb, ct_b

In [132]:
# Table parameters: q, delta, m, n, k, k_prime
table_params = [
    (2,  1, 38, 38, 30, 1125),
    (8,  1, 20, 20, 14, 270),
    (2, 1, 34, 34, 24, 800),
    (16, 1, 17, 17, 11, 183),
    (2,  1, 32, 32, 18, 564),
    (2, 2, 46, 46, 30, 1360),
     (8, 2, 30, 30, 18, 528),
]

# Table Header
# Order: q | delta | m=n | k | k' | t_pub | Cf | pk (kB) | ct (B)
header = f"{'q':<3} {'d':<3} {'m=n':<5} {'k':<4} {'k_prime':<8} {'t_pub':<6} {'Cf':<8} {'pk(kB)':<10} {'ct(B)':<8}"
print(header)
print("-" * len(header))

for q, d, m, n, k, kp in table_params:
    # Compute complexity using the synchronized function
    tp, cf, pk, ct = compute_row_complexity(q, m, n, k, kp, d)
    
    # Print row with aligned columns
    print(f"{q:<3} {d:<3} {m:<5} {k:<4} {kp:<8} {tp:<6} {cf:<8.1f} {pk:<10.2f} {ct:<8.1f}")

q   d   m=n   k    k_prime  t_pub  Cf       pk(kB)     ct(B)   
---------------------------------------------------------------
2   1   38    30   1125     4      130.9    44.86      39.9    
8   1   20    14   270      3      134.2    13.16      48.8    
2   1   34    24   800      5      130.1    35.60      44.5    
16  1   17    11   183      3      141.1    9.70       53.0    
2   1   32    18   564      7      136.5    32.43      57.5    
2   2   46    30   1360     4      131.3    128.52     94.5    
8   2   30    18   528      3      144.9    73.66      139.5   


In [138]:
# Table header for LaTeX
print("\\begin{table}[h]")
print("\\centering")
print("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|}")
print("\\hline")
print("$q$ & $\\delta$ & $m=n$ & $k$ & $k'$ & $t_{\\text{pub}}$ & $C_f$ & pk (kB) & ct (B) \\\\ \\hline")

for q, d, m, n, k, kp in table_params:
    # Compute the values
    tp, cf, pk, ct = compute_row_complexity(q, m, n, k, kp, d)
    
    # Format each row for LaTeX
    # & is the column separator, \\ is the line break
    row = f"{q} & {d} & {m} & {k} & {kp} & {tp} & {cf:.1f} & {pk:.2f} & {ct:.1f} \\\\"
    print(row)

print("\\hline")
print("\\end{tabular}")
print("\\caption{Complexity and parameters for LGS-Niederreiter}")
print("\\label{tab:complexity_results}")
print("\\end{table}")

\begin{table}[h]
\centering
\begin{tabular}{|c|c|c|c|c|c|c|c|c|}
\hline
$q$ & $\delta$ & $m=n$ & $k$ & $k'$ & $t_{\text{pub}}$ & $C_f$ & pk (kB) & ct (B) \\ \hline
2 & 1 & 59 & 49 & 2881 & 5 & 258.4 & 216.07 & 75.0 \\
2 & 1 & 47 & 31 & 1434 & 8 & 259.1 & 138.92 & 96.9 \\
2 & 1 & 49 & 35 & 1695 & 7 & 256.7 & 149.58 & 88.2 \\
8 & 1 & 27 & 17 & 447 & 5 & 264.5 & 47.27 & 105.8 \\
16 & 1 & 24 & 14 & 324 & 5 & 275.1 & 40.82 & 126.0 \\
2 & 2 & 78 & 62 & 4801 & 4 & 261.0 & 769.96 & 160.4 \\
8 & 2 & 44 & 32 & 1388 & 3 & 268.2 & 285.23 & 205.5 \\
\hline
\end{tabular}
\caption{Complexity and parameters for LGS-Niederreiter}
\label{tab:complexity_results}
\end{table}


In [145]:
# Table data: q, delta, m, n, k, k_prime
table_params = [
    #(2, 1, 47, 47, 37, 1724),
    (8, 1, 27, 27, 21, 557),
    (16, 1, 23, 23, 17, 381),
    (2, 1, 39, 39, 23, 880),
    (16, 1, 21, 21, 11, 221),
    (2, 2, 62, 62, 46, 2822),
    (8, 2, 37, 37, 25, 910),
    (16, 2, 35, 35, 19, 650),
]

# Table Header
# Order: q | delta | m=n | k | k' | t_pub | Cf | pk (kB) | ct (B)
header = f"{'q':<3} {'d':<3} {'m=n':<5} {'k':<4} {'k_prime':<8} {'t_pub':<6} {'Cf':<8} {'pk(kB)':<10} {'ct(B)':<8}"
print(header)
print("-" * len(header))

for q, d, m, n, k, kp in table_params:
    # Compute complexity using the synchronized function
    tp, cf, pk, ct = compute_row_complexity(q, m, n, k, kp, d)
    
    # Print row with aligned columns
    print(f"{q:<3} {d:<3} {m:<5} {k:<4} {kp:<8} {tp:<6} {cf:<8.1f} {pk:<10.2f} {ct:<8.1f}")
    


q   d   m=n   k    k_prime  t_pub  Cf       pk(kB)     ct(B)   
---------------------------------------------------------------
8   1   27    21   557      3      198.9    35.93      64.5    
16  1   23    17   381      3      212.3    28.19      74.0    
2   1   39    23   880      8      194.8    70.51      80.1    
16  1   21    11   221      5      212.8    24.31      110.0   
2   2   62    46   2822     4      196.0    360.51     127.8   
8   2   37    25   910      3      203.0    156.63     172.1   
16  2   35    19   650      4      210.3    186.88     287.5   


In [146]:
# Table header for LaTeX
print("\\begin{table}[h]")
print("\\centering")
print("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|}")
print("\\hline")
print("$q$ & $\\delta$ & $m=n$ & $k$ & $k'$ & $t_{\\text{pub}}$ & $C_f$ & pk (kB) & ct (B) \\\\ \\hline")

for q, d, m, n, k, kp in table_params:
    # Compute the values
    tp, cf, pk, ct = compute_row_complexity(q, m, n, k, kp, d)
    
    # Format each row for LaTeX
    # & is the column separator, \\ is the line break
    row = f"{q} & {d} & {m} & {k} & {kp} & {tp} & {cf:.1f} & {pk:.2f} & {ct:.1f} \\\\"
    print(row)

print("\\hline")
print("\\end{tabular}")
print("\\caption{Complexity and parameters for LGS-Niederreiter}")
print("\\label{tab:complexity_results}")
print("\\end{table}")

\begin{table}[h]
\centering
\begin{tabular}{|c|c|c|c|c|c|c|c|c|}
\hline
$q$ & $\delta$ & $m=n$ & $k$ & $k'$ & $t_{\text{pub}}$ & $C_f$ & pk (kB) & ct (B) \\ \hline
8 & 1 & 27 & 21 & 557 & 3 & 198.9 & 35.93 & 64.5 \\
16 & 1 & 23 & 17 & 381 & 3 & 212.3 & 28.19 & 74.0 \\
2 & 1 & 39 & 23 & 880 & 8 & 194.8 & 70.51 & 80.1 \\
16 & 1 & 21 & 11 & 221 & 5 & 212.8 & 24.31 & 110.0 \\
2 & 2 & 62 & 46 & 2822 & 4 & 196.0 & 360.51 & 127.8 \\
8 & 2 & 37 & 25 & 910 & 3 & 203.0 & 156.63 & 172.1 \\
16 & 2 & 35 & 19 & 650 & 4 & 210.3 & 186.88 & 287.5 \\
\hline
\end{tabular}
\caption{Complexity and parameters for LGS-Niederreiter}
\label{tab:complexity_results}
\end{table}


In [147]:
# Table data: q, delta, m, n, k, k_prime
table_params = [
    (2, 1, 59, 59, 49, 2881),
    (2, 1, 47, 47, 31, 1434),
    (2, 1, 49, 49, 35, 1695),
    (8, 1, 27, 27, 17, 447),
    (16, 1, 24, 24, 14, 324),
    (2, 2, 78, 78, 62, 4801),
    (8, 2, 44, 44, 32, 1388),
]

# Table Header
# Order: q | delta | m=n | k | k' | t_pub | Cf | pk (kB) | ct (B)
header = f"{'q':<3} {'d':<3} {'m=n':<5} {'k':<4} {'k_prime':<8} {'t_pub':<6} {'Cf':<8} {'pk(kB)':<10} {'ct(B)':<8}"
print(header)
print("-" * len(header))

for q, d, m, n, k, kp in table_params:
    # Compute complexity using the synchronized function
    tp, cf, pk, ct = compute_row_complexity(q, m, n, k, kp, d)
    
    # Print row with aligned columns
    print(f"{q:<3} {d:<3} {m:<5} {k:<4} {kp:<8} {tp:<6} {cf:<8.1f} {pk:<10.2f} {ct:<8.1f}")
    

q   d   m=n   k    k_prime  t_pub  Cf       pk(kB)     ct(B)   
---------------------------------------------------------------
2   1   59    49   2881     5      258.4    216.07     75.0    
2   1   47    31   1434     8      259.1    138.92     96.9    
2   1   49    35   1695     7      256.7    149.58     88.2    
8   1   27    17   447      5      264.5    47.27      105.8   
16  1   24    14   324      5      275.1    40.82      126.0   
2   2   78    62   4801     4      261.0    769.96     160.4   
8   2   44    32   1388     3      268.2    285.23     205.5   


In [148]:
# Table header for LaTeX
print("\\begin{table}[h]")
print("\\centering")
print("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|}")
print("\\hline")
print("$q$ & $\\delta$ & $m=n$ & $k$ & $k'$ & $t_{\\text{pub}}$ & $C_f$ & pk (kB) & ct (B) \\\\ \\hline")

for q, d, m, n, k, kp in table_params:
    # Compute the values
    tp, cf, pk, ct = compute_row_complexity(q, m, n, k, kp, d)
    
    # Format each row for LaTeX
    # & is the column separator, \\ is the line break
    row = f"{q} & {d} & {m} & {k} & {kp} & {tp} & {cf:.1f} & {pk:.2f} & {ct:.1f} \\\\"
    print(row)

print("\\hline")
print("\\end{tabular}")
print("\\caption{Complexity and parameters for LGS-Niederreiter}")
print("\\label{tab:complexity_results}")
print("\\end{table}")

\begin{table}[h]
\centering
\begin{tabular}{|c|c|c|c|c|c|c|c|c|}
\hline
$q$ & $\delta$ & $m=n$ & $k$ & $k'$ & $t_{\text{pub}}$ & $C_f$ & pk (kB) & ct (B) \\ \hline
2 & 1 & 59 & 49 & 2881 & 5 & 258.4 & 216.07 & 75.0 \\
2 & 1 & 47 & 31 & 1434 & 8 & 259.1 & 138.92 & 96.9 \\
2 & 1 & 49 & 35 & 1695 & 7 & 256.7 & 149.58 & 88.2 \\
8 & 1 & 27 & 17 & 447 & 5 & 264.5 & 47.27 & 105.8 \\
16 & 1 & 24 & 14 & 324 & 5 & 275.1 & 40.82 & 126.0 \\
2 & 2 & 78 & 62 & 4801 & 4 & 261.0 & 769.96 & 160.4 \\
8 & 2 & 44 & 32 & 1388 & 3 & 268.2 & 285.23 & 205.5 \\
\hline
\end{tabular}
\caption{Complexity and parameters for LGS-Niederreiter}
\label{tab:complexity_results}
\end{table}
